# Reverse-engineering the Ietswaart RNA lifecycle labels

This notebook determines how `rna-lifecycle-ietswaart` differs from the independently prepared `rna-loc-ietswaart` dataset. Both datasets originate from the same K562 direct-RNA sequencing experiment, but they encode different labels.

The analysis uses immutable Hugging Face revisions so it does not depend on personal filesystem paths. It distinguishes:

- **observed facts**, which can be checked directly from the published parquet files;
- **recovered processing**, which follows the localization preprocessing notebook; and
- **inference**, where the continuous pre-binarization coverage values are no longer available.

The central question is whether the lifecycle labels came from independent kinetic-rate measurements or from re-thresholding the normalized compartment abundances used for the localization dataset.

In [1]:
from __future__ import annotations

from collections import defaultdict
from functools import reduce
import hashlib
import os
from pathlib import Path
import re

from huggingface_hub import hf_hub_download
import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 120)

LABEL_NAMES = ("Chromatin", "Cytoplasm", "Polysome")
COMPARTMENT_CODES = ("chr", "cyto", "poly")

# Final user-prepared localization artifact.
LOC_REPO = "morrislab/rna-loc-ietswaart"
LOC_REVISION = "21ffad7a621e7a5b27829053845efad1ca4171e5"
LOC_FILENAME = "rna-loc-ietswaart.parquet"

# Current lifecycle artifact. This revision only reordered columns relative to
# the first upload; its row values and targets are unchanged.
LIFECYCLE_REPO = "morrislab/rna-lifecycle-ietswaart"
LIFECYCLE_REVISION = "1c1e15f85b05cb1e8b2d044f33f8467eaa3c1e8a"
LIFECYCLE_FILENAME = "ietswaart_processed.parquet"


def resolve_parquet(
    env_var: str,
    repo_id: str,
    revision: str,
    filename: str,
) -> Path:
    """Use an explicit local override, otherwise fetch an immutable Hub revision."""
    override = os.environ.get(env_var)
    if override:
        path = Path(override).expanduser().resolve()
        if not path.is_file():
            raise FileNotFoundError(f"{env_var} does not point to a file: {path}")
        return path

    return Path(
        hf_hub_download(
            repo_id=repo_id,
            filename=filename,
            repo_type="dataset",
            revision=revision,
        )
    )


def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


loc_path = resolve_parquet(
    "IETSWAART_LOC_PARQUET", LOC_REPO, LOC_REVISION, LOC_FILENAME
)
lifecycle_path = resolve_parquet(
    "IETSWAART_LIFECYCLE_PARQUET",
    LIFECYCLE_REPO,
    LIFECYCLE_REVISION,
    LIFECYCLE_FILENAME,
)

pd.DataFrame(
    [
        {
            "artifact": "localization",
            "repository": LOC_REPO,
            "revision": LOC_REVISION,
            "filename": LOC_FILENAME,
            "sha256": sha256(loc_path),
        },
        {
            "artifact": "lifecycle",
            "repository": LIFECYCLE_REPO,
            "revision": LIFECYCLE_REVISION,
            "filename": LIFECYCLE_FILENAME,
            "sha256": sha256(lifecycle_path),
        },
    ]
)

,artifact,repository,revision,filename,sha256
0,localization,morrislab/rna-loc-ietswaart,21ffad7a621e7a5b27829053845efad1ca4171e5,rna-loc-ietswaart.parquet,5a8f8b0cec5c1097023ee635fac8f54b4957daf88e9396a00a1ede442163de49
1,lifecycle,morrislab/rna-lifecycle-ietswaart,1c1e15f85b05cb1e8b2d044f33f8467eaa3c1e8a,ietswaart_processed.parquet,5706b21ff8f1e1005a42339829bc40b84cb0afccd34f3d5d2273369e8dec4746


## 1. Load the two published artifacts

In [2]:
loc_df = pd.read_parquet(loc_path)
lifecycle_df = pd.read_parquet(lifecycle_path)

pd.DataFrame(
    [
        {
            "artifact": "localization",
            "rows": len(loc_df),
            "columns": list(loc_df.columns),
        },
        {
            "artifact": "lifecycle",
            "rows": len(lifecycle_df),
            "columns": list(lifecycle_df.columns),
        },
    ]
)

,artifact,rows,columns
0,localization,10043,"[transcript_id, gene, chromosome, sequence, cds, splice, target]"
1,lifecycle,10043,"[splice, cds, sequence, chromosome, gene, transcript_id, target]"


## 2. Establish whether the lifecycle table reused the localization table

Array-valued columns need element-wise array comparison rather than ordinary pandas equality.

In [3]:
FEATURE_COLUMNS = (
    "transcript_id",
    "gene",
    "chromosome",
    "sequence",
    "cds",
    "splice",
)
ARRAY_COLUMNS = {"cds", "splice"}

if len(loc_df) != len(lifecycle_df):
    raise AssertionError("The artifacts do not contain the same number of rows")

identity_rows = []
for column in FEATURE_COLUMNS:
    if column in ARRAY_COLUMNS:
        equal = np.fromiter(
            (
                np.array_equal(left, right)
                for left, right in zip(loc_df[column], lifecycle_df[column])
            ),
            dtype=bool,
            count=len(loc_df),
        )
    else:
        equal = loc_df[column].to_numpy() == lifecycle_df[column].to_numpy()

    identity_rows.append(
        {
            "column": column,
            "equal_rows": int(equal.sum()),
            "total_rows": len(equal),
            "all_equal_in_same_order": bool(equal.all()),
        }
    )

identity_audit = pd.DataFrame(identity_rows)
display(identity_audit)
assert identity_audit["all_equal_in_same_order"].all()

,column,equal_rows,total_rows,all_equal_in_same_order
0,transcript_id,10043,10043,True
1,gene,10043,10043,True
2,chromosome,10043,10043,True
3,sequence,10043,10043,True
4,cds,10043,10043,True
5,splice,10043,10043,True


**Observed fact:** all 10,043 rows, including transcript order, duplicate rows, sequences, CDS tracks, and splice tracks, are identical. The lifecycle parquet therefore reused the completed localization feature table and replaced only `target`; it was not independently assembled from a kinetic-rate dataset.

## 3. Compare label distributions

In [4]:
loc_target = np.stack(loc_df["target"].to_numpy()).astype(np.int64)
lifecycle_target = np.stack(lifecycle_df["target"].to_numpy()).astype(np.int64)


def target_summary(name: str, target: np.ndarray) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "artifact": name,
            "label": LABEL_NAMES,
            "positive_count": target.sum(axis=0),
            "positive_fraction": target.mean(axis=0),
        }
    )


display(
    pd.concat(
        [
            target_summary("localization", loc_target),
            target_summary("lifecycle", lifecycle_target),
        ],
        ignore_index=True,
    )
)

patterns, counts = np.unique(lifecycle_target, axis=0, return_counts=True)
pattern_table = pd.DataFrame(
    {
        "target": [tuple(row) for row in patterns],
        "count": counts,
    }
)
display(pattern_table)

,artifact,label,positive_count,positive_fraction
0,localization,Chromatin,2801,0.278901
1,localization,Cytoplasm,5517,0.549338
2,localization,Polysome,6715,0.668625
3,lifecycle,Chromatin,3314,0.329981
4,lifecycle,Cytoplasm,3314,0.329981
5,lifecycle,Polysome,3312,0.329782


,target,count
0,"(0, 0, 0)",3061
1,"(0, 0, 1)",1086
2,"(0, 1, 0)",1080
3,"(0, 1, 1)",1502
4,"(1, 0, 0)",1858
5,"(1, 0, 1)",724
6,"(1, 1, 0)",732


The lifecycle positive fractions are approximately 33%, not 20%. The public data card's claim that labels are the bottom 20th percentile is inconsistent with the parquet.

## 4. Test the relationship between localization and lifecycle labels

The localization notebook labels a compartment when its row-normalized coverage proportion is at least `0.33`. If lifecycle labels represent the lower tail of those same proportions, every lifecycle-positive row must be localization-negative for the corresponding compartment.

In [5]:
relationship_rows = []
for index, label in enumerate(LABEL_NAMES):
    overlap = (loc_target[:, index] == 1) & (lifecycle_target[:, index] == 1)
    relationship_rows.append(
        {
            "label": label,
            "localization_positive": int(loc_target[:, index].sum()),
            "lifecycle_positive": int(lifecycle_target[:, index].sum()),
            "positive_in_both": int(overlap.sum()),
            "lifecycle_positive_but_localization_positive": int(overlap.sum()),
        }
    )

relationship_audit = pd.DataFrame(relationship_rows)
display(relationship_audit)
assert (relationship_audit["positive_in_both"] == 0).all()

,label,localization_positive,lifecycle_positive,positive_in_both,lifecycle_positive_but_localization_positive
0,Chromatin,2801,3314,0,0
1,Cytoplasm,5517,3314,0,0
2,Polysome,6715,3312,0,0


**Observed fact:** there are zero violations across 30,129 label comparisons. A lifecycle-positive is always a localization-negative in the same compartment. This exact nesting is the expected result of applying a lower-tail cutoff to the same normalized coverage values and is not expected from an independent set of TimeLapse-seq kinetic rates.

## 5. Recover the quantile used for lifecycle labels

For a strictly ordered sample, `values < np.quantile(values, q)` selects the number of observations shown below. Ties at the cutoff can reduce that count, which explains why the polysome class has two fewer positives.

In [6]:
def expected_strict_below_count(sample_size: int, quantile: float) -> int:
    """Count below NumPy's linear quantile when values around it are distinct."""
    position = (sample_size - 1) * quantile
    if float(position).is_integer():
        return int(position)
    return int(np.floor(position)) + 1


candidate_quantiles = (0.20, 0.25, 0.30, 0.33, 1 / 3)
observed_counts = lifecycle_target.sum(axis=0)
quantile_audit = pd.DataFrame(
    [
        {
            "candidate_quantile": quantile,
            "expected_count_without_ties": expected_strict_below_count(
                len(lifecycle_df), quantile
            ),
            "absolute_error_chromatin": abs(
                expected_strict_below_count(len(lifecycle_df), quantile)
                - observed_counts[0]
            ),
            "absolute_error_cytoplasm": abs(
                expected_strict_below_count(len(lifecycle_df), quantile)
                - observed_counts[1]
            ),
            "absolute_error_polysome": abs(
                expected_strict_below_count(len(lifecycle_df), quantile)
                - observed_counts[2]
            ),
        }
        for quantile in candidate_quantiles
    ]
)
quantile_audit

,candidate_quantile,expected_count_without_ties,absolute_error_chromatin,absolute_error_cytoplasm,absolute_error_polysome
0,0.200000,2009,1305,1305,1303
1,0.250000,2511,803,803,801
2,0.300000,3013,301,301,299
3,0.330000,3314,0,0,2
4,0.333333,3348,34,34,36


**Exact recovered rule:** the processor used the strict lower 33rd percentile independently for each compartment. With 10,043 rows, `q=0.33` produces `(3314, 3314, 3312)` positives, exactly matching the published artifact. Neither `q=0.20` nor the mathematical value `1/3` matches.

The target operation is:

```python
cutoffs = np.quantile(normalized_compartment_coverage, 0.33, axis=0)
target = (normalized_compartment_coverage < cutoffs).astype(np.int64)
```

## 6. Reconstruct the continuous compartment proportions

The following function captures the relevant processing from the localization notebook. It expects the transcript-level tab-separated outputs produced after aligning/quantifying each FASTQ; the eight FASTQ files alone do not contain the `cov`, reference-transcript, or `class_code` columns needed here.

Important behavior retained from the original processing:

1. Require positive coverage in each replicate before pairing replicates.
2. Inner-join replicate pairs and retain exact (`class_code == "="`) reference matches.
3. Average replicate coverage independently for `chr`, `cyto`, `poly`, and `total`.
4. Outer-join compartments, fill absent subcellular coverage with zero, require positive total coverage, and require at least one observed subcellular compartment.
5. Row-normalize `chr`, `cyto`, and `poly` coverage so each target vector sums to one.

In [7]:
RAW_JOIN_COLUMNS = (
    "ref_gene_id",
    "ref_id",
    "class_code",
    "num_exons",
    "len",
)
OUTPUT_JOIN_COLUMNS = ("gene", "transcript_id", "num_exons", "length")
DROP_COLUMNS = ("qry_id", "parent gene iso num", "sample_id")


def reconstruct_continuous_proportions(tracking_dir: str | Path) -> pd.DataFrame:
    """Recreate normalized chr/cyto/poly proportions from replicate tables."""
    tracking_dir = Path(tracking_dir).expanduser().resolve()
    if not tracking_dir.is_dir():
        raise NotADirectoryError(tracking_dir)

    grouped_files: dict[str, list[Path]] = defaultdict(list)
    pattern = re.compile(r"^K562_(chr|cyto|poly|total)_")
    fastq_suffixes = (".fastq", ".fq", ".fastq.gz", ".fq.gz")
    for path in tracking_dir.iterdir():
        match = pattern.match(path.name)
        if path.is_file() and match and not path.name.endswith(fastq_suffixes):
            grouped_files[match.group(1)].append(path)

    missing_or_unpaired = {
        compartment: len(grouped_files.get(compartment, []))
        for compartment in (*COMPARTMENT_CODES, "total")
        if len(grouped_files.get(compartment, [])) != 2
    }
    if missing_or_unpaired:
        raise ValueError(
            "Expected exactly two transcript-level replicate tables per group; "
            f"found {missing_or_unpaired}. FASTQ files must first be aligned and quantified."
        )

    merged_by_compartment = {}
    for compartment in (*COMPARTMENT_CODES, "total"):
        replicate_frames = []
        for path in sorted(grouped_files[compartment]):
            frame = pd.read_csv(path, sep="\t")
            required_columns = set(RAW_JOIN_COLUMNS) | set(DROP_COLUMNS) | {"cov"}
            missing_columns = required_columns.difference(frame.columns)
            if missing_columns:
                raise ValueError(
                    f"{path} is not a transcript-level coverage table; "
                    f"missing columns: {sorted(missing_columns)}"
                )
            frame = frame.loc[frame["cov"] > 0].drop(columns=list(DROP_COLUMNS))
            replicate_frames.append(frame)

        merged = replicate_frames[0].merge(
            replicate_frames[1], on=list(RAW_JOIN_COLUMNS), how="inner"
        )
        merged = merged.loc[merged["class_code"] == "="].drop(columns="class_code")
        merged[f"coverage_{compartment}"] = merged[["cov_x", "cov_y"]].mean(
            axis=1
        )
        merged = merged.drop(columns=["cov_x", "cov_y"]).rename(
            columns={
                "ref_gene_id": "gene",
                "ref_id": "transcript_id",
                "len": "length",
            }
        )
        merged_by_compartment[compartment] = merged.reset_index(drop=True)

    combined = reduce(
        lambda left, right: left.merge(
            right, on=list(OUTPUT_JOIN_COLUMNS), how="outer"
        ),
        [
            merged_by_compartment[compartment]
            for compartment in (*COMPARTMENT_CODES, "total")
        ],
    )

    coverage_columns = [f"coverage_{name}" for name in COMPARTMENT_CODES]
    combined[coverage_columns] = combined[coverage_columns].fillna(0)
    combined = combined.loc[combined["coverage_total"] > 0].drop(
        columns="coverage_total"
    )
    combined = combined.loc[combined[coverage_columns].sum(axis=1) > 0].copy()
    combined[coverage_columns] = combined[coverage_columns].div(
        combined[coverage_columns].sum(axis=1), axis=0
    )
    combined["continuous_target"] = list(
        combined[coverage_columns].to_numpy(dtype=np.float64)
    )
    return combined.reset_index(drop=True)


## 7. Convert continuous proportions into lifecycle labels

In [8]:
def make_lifecycle_labels(
    continuous_target: pd.Series | np.ndarray,
    quantile: float = 0.33,
) -> tuple[np.ndarray, np.ndarray]:
    """Label the strict lower quantile independently for each compartment."""
    if isinstance(continuous_target, pd.Series):
        values = np.stack(continuous_target.to_numpy())
    else:
        values = np.asarray(continuous_target)

    if values.ndim != 2 or values.shape[1] != len(LABEL_NAMES):
        raise ValueError(
            f"Expected an (n, {len(LABEL_NAMES)}) matrix, got {values.shape}"
        )
    if not np.isfinite(values).all():
        raise ValueError("Continuous targets contain non-finite values")

    cutoffs = np.quantile(values, quantile, axis=0)
    labels = (values < cutoffs).astype(np.int64)
    return labels, cutoffs


def replace_targets(
    localization_template: pd.DataFrame,
    labels: np.ndarray,
) -> pd.DataFrame:
    """Reuse the localization features exactly and replace only the target."""
    if len(localization_template) != len(labels):
        raise ValueError(
            f"Template has {len(localization_template)} rows but labels have {len(labels)}"
        )

    output = localization_template.copy()
    output["target"] = list(np.asarray(labels, dtype=np.int64))
    return output[
        [
            "transcript_id",
            "gene",
            "chromosome",
            "sequence",
            "cds",
            "splice",
            "target",
        ]
    ]

## 8. Exact reconstruction check

The exact eight `wf-transcriptomes` tables were recovered and stored in the checksum-pinned top-level LFS archive. This cell rebuilds the complete dataframe and compares every ordered scalar and array value with the published lifecycle parquet.

In [9]:
from tempfile import TemporaryDirectory

from mrna_bench.datasets.rna_lifecycle_ietswaart import (
    _add_sequence_features,
    _extract_source_tables,
    _make_lifecycle_targets,
    _normalized_compartment_coverage,
)

repo_root = Path.cwd()
while not (repo_root / "pyproject.toml").is_file():
    if repo_root == repo_root.parent:
        raise FileNotFoundError("Could not locate the mRNABench repository root")
    repo_root = repo_root.parent
source_archive = repo_root / "resources" / "ietswaart_wf_transcript_tables.tar.gz"

with TemporaryDirectory() as temporary_dir:
    table_paths = _extract_source_tables(source_archive, Path(temporary_dir))
    continuous_df = _normalized_compartment_coverage(table_paths)
    reconstructed_labels = _make_lifecycle_targets(continuous_df)
    reconstructed_df = _add_sequence_features(
        continuous_df,
        reconstructed_labels,
    )

comparison_rows = []
for column in lifecycle_df.columns:
    if column in {"cds", "splice", "target"}:
        equal = np.fromiter(
            (
                np.array_equal(left, right)
                for left, right in zip(reconstructed_df[column], lifecycle_df[column])
            ),
            dtype=bool,
            count=len(lifecycle_df),
        )
    else:
        equal = reconstructed_df[column].to_numpy() == lifecycle_df[column].to_numpy()
    comparison_rows.append(
        {
            "column": column,
            "equal_rows": int(equal.sum()),
            "total_rows": len(equal),
            "all_equal": bool(equal.all()),
        }
    )

exact_audit = pd.DataFrame(comparison_rows)
display(exact_audit)
assert exact_audit["all_equal"].all()

,column,equal_rows,total_rows,all_equal
0,splice,10043,10043,True
1,cds,10043,10043,True
2,sequence,10043,10043,True
3,chromosome,10043,10043,True
4,gene,10043,10043,True
5,transcript_id,10043,10043,True
6,target,10043,10043,True


## Conclusion

### Recovered with direct evidence

1. `rna-lifecycle-ietswaart` reused the final 10,043-row `rna-loc-ietswaart` feature table exactly. All transcript identifiers, duplicated rows, genes, chromosomes, sequences, CDS tracks, splice tracks, and row ordering match.
2. Only `target` changed.
3. For each compartment, every lifecycle-positive row is a localization-negative row. There are no exceptions.
4. Lifecycle class counts are `(3314, 3314, 3312)`, approximately 33% of 10,043 per compartment—not 20%.

### Exact processing recipe

The lifecycle processor took the same row-normalized direct-RNA coverage proportions used for localization and relabeled each compartment independently as depleted when its proportion was strictly below that compartment's 33rd percentile:

```python
cutoffs = np.quantile(proportions, 0.33, axis=0)
lifecycle_target = (proportions < cutoffs).astype(np.int64)
```

The strict comparison and tied polysome values produce two fewer polysome positives. This transformation also explains why lifecycle positives are always excluded by the localization rule `proportion >= 0.33`.

### Exact reproduction result

The recovered transcript tables reproduce all 10,043 ordered rows, metadata values, sequences, CDS tracks, splice tracks, and 30,129 target bits exactly. Physical parquet bytes depend on the pandas/pyarrow writer versions; the published file used pandas 2.2.2 and pyarrow 20.0.0.

The public lifecycle data card should not describe these labels as the bottom 20th percentile of independent RNA-flow rates. They are exact lower-tertile depletion labels derived from the same direct-RNA compartment abundance data as the localization dataset.